# H3 Resolutions — A Visual Guide
### Companion notebook · how big is a hexagon, and which size should I use?

The hands-on lab picks **resolution 7** and moves on. This notebook is the part we skipped:
what the sixteen resolutions actually look like, how they fit inside each other, and how to
choose one for your own data.

Five questions, answered with pictures:

1. **How big is each resolution?** — all 16, with real-world comparisons
2. **What does that look like on a map?** — one place, drawn five ways
3. **How do resolutions nest?** — and the surprise that catches people out
4. **What happens to real data?** — the same dataset aggregated five ways
5. **So which one do I pick?** — the rule of thumb, and how to test it

Run the cells in order. Every cell opens with a comment block explaining what it does.

---
## Part 0 · Setup

In [ ]:
# ==========================================================================
# STEP 1 · Install h3 and import everything.
# --------------------------------------------------------------------------
# The version is pinned to match the hands-on lab. h3-py v4 renamed every
# function from v3, so a v3 tutorial will not run here.
# ==========================================================================
!pip install -q h3==4.5.0

In [ ]:
# ==========================================================================
# STEP 2 · Imports and one shared helper.
# --------------------------------------------------------------------------
# draw_cells() is used by every map in this notebook. It takes a list of H3
# cell ids and draws each one as a polygon on a matplotlib axis.
#
# The important detail: cell_to_boundary() returns (LATITUDE, LONGITUDE)
# pairs, but matplotlib wants (x, y) = (longitude, latitude). Getting this
# backwards is the single most common H3 mistake, and it silently produces a
# map of the Indian Ocean.
# ==========================================================================
import math
import numpy as np
import pandas as pd
import h3
import folium
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

def draw_cells(ax, cells, facecolor="none", edgecolor="#333", lw=0.6, alpha=1.0):
    """Draw H3 cells as polygons on a matplotlib axis."""
    for c in cells:
        boundary = h3.cell_to_boundary(c)                  # ((lat, lng), ...)
        xy = [(lng, lat) for lat, lng in boundary]         # -> (x, y)
        ax.add_patch(MplPolygon(xy, closed=True, facecolor=facecolor,
                                edgecolor=edgecolor, linewidth=lw, alpha=alpha))

print("h3 version:", h3.__version__)

---
## Part 1 · All sixteen resolutions

H3 is a **hierarchy**, not a single grid. Resolution 0 divides the planet into 122 enormous
cells; each step finer divides each cell into roughly seven smaller ones. That "roughly" is
doing some work, and Part 3 explains why.

In [ ]:
# ==========================================================================
# STEP 3 · Build the reference table for all 16 resolutions.
# --------------------------------------------------------------------------
# Three functions, none of which need an actual cell -- they describe the
# resolution itself:
#   average_hexagon_edge_length(r)  side length
#   average_hexagon_area(r)         area
#   get_num_cells(r)                how many cells cover the Earth
#
# "average" is not a hedge. Cells genuinely vary in size within a resolution;
# Part 5 measures the spread.
# ==========================================================================
SIZE_OF = {
    0: "~ the European Union", 1: "~ Ukraine", 2: "~ Austria", 3: "~ Qatar",
    4: "~ Greater London", 5: "~ a whole city", 6: "~ a large town",
    7: "~ 1.5 Central Parks", 8: "~ 100 football pitches", 9: "~ a city block",
    10: "~ 2 football pitches", 11: "~ a large house plot", 12: "~ a tennis court",
    13: "~ a parking bay", 14: "~ a small bathroom", 15: "~ a doormat",
}

table = pd.DataFrame([{
    "res": r,
    "edge": h3.average_hexagon_edge_length(r, unit="km"),
    "area_km2": h3.average_hexagon_area(r, unit="km^2"),
    "cells_on_earth": h3.get_num_cells(r),
    "roughly": SIZE_OF[r],
} for r in range(16)])

def human_edge(km):
    return f"{km*1000:,.0f} m" if km < 1 else f"{km:,.1f} km"

def human_area(km2):
    return f"{km2*1e6:,.1f} m2" if km2 < 0.01 else f"{km2:,.2f} km2"

show = table.copy()
show["edge"] = show["edge"].map(human_edge)
show["area"] = show["area_km2"].map(human_area)
show["cells_on_earth"] = show["cells_on_earth"].map(lambda v: f"{v:,}")
show[["res", "edge", "area", "cells_on_earth", "roughly"]]

In [ ]:
# ==========================================================================
# STEP 4 · The same table as a picture.
# --------------------------------------------------------------------------
# A log scale is essential here: resolution 0 cells are about 4.4 TRILLION
# times the area of resolution 15 cells. On a linear axis every bar except
# the first would be invisible.
#
# The line is almost perfectly straight, which is the real message -- each
# step is a constant ~7x change in area. Choosing a resolution is choosing an
# order of magnitude, not fine-tuning a number.
# ==========================================================================
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(table["res"], table["area_km2"], "o-", color="#c1440e")
ax.set_yscale("log")
ax.set_xlabel("resolution"); ax.set_ylabel("average cell area, km2 (log scale)")
ax.set_title("Each step finer divides the area by about 7")
ax.set_xticks(range(16)); ax.grid(alpha=.3, which="both")

for r in (0, 7, 9, 15):
    ax.annotate(f"res {r}\n{SIZE_OF[r]}",
                (r, table.loc[r, "area_km2"]),
                textcoords="offset points", xytext=(8, 8), fontsize=8)
plt.show()

Two things worth committing to memory:

- **Resolution 9 ≈ 0.1 km² ≈ a city block.** It is the most useful anchor in the whole table —
  most analytics work lands within a step or two of it.
- **Resolution 0 has 122 cells, not 120.** 110 hexagons and **12 pentagons**. You cannot tile a
  sphere with hexagons alone, so H3 places twelve pentagons, positioned so their centres fall in
  open water where they inconvenience almost nobody. Part 5 comes back to this.

---
## Part 2 · One place, drawn five ways

Numbers in a table are abstract. Here is the *same patch of ground* — about 10 km across, centred
on Perungudi in Chennai — covered at five different resolutions.

In [ ]:
# ==========================================================================
# STEP 5 · Draw the same fixed map extent at five resolutions.
# --------------------------------------------------------------------------
# The trick that makes the comparison honest: the axis limits are IDENTICAL
# in all five panels. Only the grid changes. So the panels show granularity,
# not zoom.
#
# For each resolution we take the cell containing our anchor point and expand
# outward with grid_disk until the cells more than cover the box, then let
# matplotlib clip whatever falls outside.
# ==========================================================================
ANCHOR = (12.9698, 80.2437)          # Perungudi, Chennai
HALF_KM = 5.0                        # half-width of the box

dlat = HALF_KM / 110.574                                     # km -> degrees lat
dlon = HALF_KM / (111.320 * math.cos(math.radians(ANCHOR[0])))

fig, axes = plt.subplots(1, 5, figsize=(19, 4.2))
for ax, res in zip(axes, (5, 6, 7, 8, 9)):
    centre = h3.latlng_to_cell(*ANCHOR, res)
    edge = h3.average_hexagon_edge_length(res, unit="km")
    # cover the box CORNERS, not just its edges, or the grid stops short and
    # leaves white wedges in the corners of the finer panels
    k = math.ceil(HALF_KM * math.sqrt(2) / (1.5 * edge)) + 1
    cells = h3.grid_disk(centre, k)

    draw_cells(ax, cells, facecolor="#4fd1c5", edgecolor="#0f766e", lw=0.5, alpha=0.25)
    draw_cells(ax, [centre], facecolor="#c1440e", edgecolor="#c1440e", lw=1.2, alpha=0.85)
    ax.plot(ANCHOR[1], ANCHOR[0], "k.", markersize=5)

    ax.set_xlim(ANCHOR[1] - dlon, ANCHOR[1] + dlon)
    ax.set_ylim(ANCHOR[0] - dlat, ANCHOR[0] + dlat)
    ax.set_aspect(1 / math.cos(math.radians(ANCHOR[0])))     # don't squash the hexagons
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"res {res}\nedge {human_edge(edge)}  ·  {len(cells)} cells drawn", fontsize=10)

fig.suptitle(f"The same 10 km box around Perungudi, Chennai — "
             f"orange = the cell containing the point", y=1.04)
plt.tight_layout(); plt.show()

In [ ]:
# ==========================================================================
# STEP 6 · The same thing on a real slippy map, with layers you can toggle.
# --------------------------------------------------------------------------
# Static panels show the size change; this shows the hexagons sitting on real
# streets. Use the layer control in the top right to switch resolutions --
# res 7 covers a suburb, res 9 covers a block, res 10 covers a building or two.
# ==========================================================================
m = folium.Map(location=list(ANCHOR), zoom_start=13, tiles="cartodbpositron")
folium.Marker(list(ANCHOR), tooltip="Perungudi, Chennai").add_to(m)

for res, colour in ((7, "#c1440e"), (8, "#2b6cb0"), (9, "#0f766e"), (10, "#7c3aed")):
    centre = h3.latlng_to_cell(*ANCHOR, res)
    edge = h3.average_hexagon_edge_length(res, unit="km")
    k = math.ceil(1.5 / (1.5 * edge))                        # ~1.5 km of context
    layer = folium.FeatureGroup(name=f"resolution {res}", show=(res == 9))
    for c in h3.grid_disk(centre, k):
        folium.Polygon(
            locations=[list(p) for p in h3.cell_to_boundary(c)],   # folium wants (lat, lng)
            color=colour, weight=1.5, fill=True,
            fill_opacity=0.35 if c == centre else 0.08,
            tooltip=f"res {res} · {c}",
        ).add_to(layer)
    layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

---
## Part 3 · How resolutions nest — and the catch

Every cell has one parent at each coarser resolution and seven children at the next finer one.
That hierarchy is baked into the id itself, which is why you can change resolution without
touching the original coordinates.

In [ ]:
# ==========================================================================
# STEP 7 · Walk one point up and down the hierarchy.
# --------------------------------------------------------------------------
# Notice the ids share a growing prefix as you go finer. The hierarchy is
# literally encoded in the id, so cell_to_parent is a bit-shift, not a lookup.
# ==========================================================================
point_cell = h3.latlng_to_cell(*ANCHOR, 9)
print(f"our point at res 9 : {point_cell}\n")

print(f"{'res':>4}  {'cell':<18}{'area km2':>12}")
for r in range(4, 10):
    c = h3.cell_to_parent(point_cell, r) if r < 9 else point_cell
    print(f"{r:>4}  {c:<18}{h3.cell_area(c, unit='km^2'):>12.4f}")

print(f"\nchildren of the res-7 ancestor at res 8: "
      f"{len(h3.cell_to_children(h3.cell_to_parent(point_cell, 7), 8))}")
print(f"grandchildren at res 9: "
      f"{len(h3.cell_to_children(h3.cell_to_parent(point_cell, 7), 9))}")

In [ ]:
# ==========================================================================
# STEP 8 · Draw the nesting -- and see the problem immediately.
# --------------------------------------------------------------------------
# Left panel:  one res-7 parent (thick outline) with its 7 res-8 children.
# Right panel: the same parent with its 49 res-9 grandchildren.
#
# Look at the edges. The children do NOT line up with the parent's boundary --
# they spill out along some edges and fall short along others. Seven hexagons
# cannot tile one larger hexagon exactly. This is not a rendering artefact.
# ==========================================================================
parent = h3.cell_to_parent(point_cell, 7)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, child_res in zip(axes, (8, 9)):
    children = h3.cell_to_children(parent, child_res)
    draw_cells(ax, children, facecolor="#4fd1c5", edgecolor="#0f766e", lw=0.8, alpha=0.35)
    draw_cells(ax, [parent], facecolor="none", edgecolor="#c1440e", lw=3)

    b = h3.cell_to_boundary(parent)
    pad = 0.004
    ax.set_xlim(min(p[1] for p in b) - pad, max(p[1] for p in b) + pad)
    ax.set_ylim(min(p[0] for p in b) - pad, max(p[0] for p in b) + pad)
    ax.set_aspect(1 / math.cos(math.radians(ANCHOR[0])))
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"one res-7 cell (orange) and its {len(children)} res-{child_res} children")
plt.tight_layout(); plt.show()

### Measuring the mismatch

The picture suggests the children don't fill the parent exactly. Let's not argue about it —
let's measure it. Scatter random points inside the parent, and count how many land in one of
its official children.

In [ ]:
# ==========================================================================
# STEP 9 · Quantify the imperfect nesting.
# --------------------------------------------------------------------------
# Method: throw random points at the parent's bounding box, keep only the ones
# that really are inside the parent at res 7, then ask whether each of those
# points also sits inside one of the parent's seven res-8 children.
#
# If nesting were exact, the answer would be 100%.
# ==========================================================================
rng = np.random.default_rng(0)
kids = set(h3.cell_to_children(parent, 8))
b = h3.cell_to_boundary(parent)
lat_lo, lat_hi = min(p[0] for p in b), max(p[0] for p in b)
lon_lo, lon_hi = min(p[1] for p in b), max(p[1] for p in b)

TARGET = 20_000
inside = tested = 0
while tested < TARGET:
    la = rng.uniform(lat_lo, lat_hi)
    lo = rng.uniform(lon_lo, lon_hi)
    if h3.latlng_to_cell(la, lo, 7) != parent:      # not actually in the parent
        continue
    tested += 1
    if h3.latlng_to_cell(la, lo, 8) in kids:
        inside += 1

print(f"points sampled inside the res-7 parent   : {tested:,}")
print(f"  ...that land in one of its 7 children  : {inside:,}  ({100*inside/tested:.1f}%)")
print(f"  ...that land OUTSIDE all of them       : {tested-inside:,}  "
      f"({100*(tested-inside)/tested:.1f}%)")
print()
print("Roughly 7% of the parent's own area belongs to cells that are not its children.")
print("The grid also twists about 19 degrees at each step, which is why the")
print("children sit at an angle to the parent.")

### Why this matters more than it sounds

`cell_to_children` is an **indexing** operation, not a **geometric** one. It answers *"which ids
are formally beneath this id?"*, not *"which cells cover this ground?"*

The practical consequence, and it is a real bug people ship:

> **Never roll counts up the hierarchy.** If you have totals per res-9 cell, do **not** sum them
> into their res-8 parents and call it a res-8 total. About 7% of the value is in the wrong
> place at every level, and the error compounds as you climb.
>
> **Re-aggregate from the raw points instead** — index the original data at the resolution you
> want. It costs one more `latlng_to_cell` pass and it is correct.

---
## Part 4 · The same data at five resolutions

Abstract grids are one thing. Here is the effect on the dataset from the hands-on lab —
20,640 California block groups — aggregated six ways.

In [ ]:
# ==========================================================================
# STEP 10 · Load the lab dataset.
# --------------------------------------------------------------------------
# Same California housing data as the hands-on lab: 20,640 census block
# groups from the 1990 US Census, each with coordinates and a median house
# value. Source and full description are in that notebook.
# Two URLs so one dead link can't break this notebook.
# ==========================================================================
LAB = ("https://raw.githubusercontent.com/litandlatte/tanacloud.com/"
       "geospatial-data-science-with-ubers-h3/labs/geospatial-data-science-with-ubers-h3")

df = None
for u in ["https://raw.githubusercontent.com/ageron/handson-ml2/master/"
          "datasets/housing/housing.csv", f"{LAB}/data/housing.csv"]:
    try:
        df = pd.read_csv(u); print("loaded from:", u); break
    except Exception as e:
        print("failed:", u, "->", type(e).__name__)

df = df.dropna(subset=["median_house_value", "latitude", "longitude"]).reset_index(drop=True)
lat_a = df["latitude"].to_numpy(float)
lon_a = df["longitude"].to_numpy(float)
val_a = df["median_house_value"].to_numpy(float)
print(f"{len(df):,} block groups")

In [ ]:
# ==========================================================================
# STEP 11 · Aggregate at each resolution and report what happens.
# --------------------------------------------------------------------------
# The two columns that decide everything:
#   pts/cell     how much evidence each cell's average is based on
#   1-pt cells   the share of cells holding a single point -- those "averages"
#                are not averages at all, they are just that one value
#
# Watch them move in opposite directions as the grid gets finer. That tension
# IS the resolution choice.
# ==========================================================================
panels = {}
rows = []
for res in (4, 5, 6, 7, 8, 9):
    cl = np.array([h3.latlng_to_cell(a, o, res) for a, o in zip(lat_a, lon_a)])
    g = pd.DataFrame({"cell": cl, "v": val_a}).groupby("cell")["v"].agg(["mean", "count"])
    panels[res] = g
    rows.append({
        "res": res,
        "edge": human_edge(h3.average_hexagon_edge_length(res, unit="km")),
        "cells used": len(g),
        "median pts/cell": int(g["count"].median()),
        "cells with 1 pt": f"{100*(g['count'] == 1).mean():.0f}%",
    })

pd.DataFrame(rows).set_index("res")

In [ ]:
# ==========================================================================
# STEP 12 · The same map at six resolutions -- across TWO extents.
# --------------------------------------------------------------------------
# Why two extents rather than one row of six? Because a res-9 hexagon drawn
# on a map of the whole state is smaller than a pixel. Showing it there would
# not prove the grid is too fine, it would only prove the picture is too
# small -- so the fine resolutions get a zoomed panel where they are legible.
#
# That is itself the lesson: RESOLUTION HAS TO MATCH THE EXTENT YOU ARE
# LOOKING AT. Resolutions 4-6 describe a state; 7-9 describe a metro area.
#
# Colour is mean house value. Only cells holding >= 3 block groups are drawn,
# because a "mean" over one or two is noise rather than signal -- which is
# also why the top-right panels start thinning out.
# ==========================================================================
CA = {"lat": (32.3, 42.2), "lon": (-124.6, -113.9)}
BAY = {"lat": (37.15, 38.15), "lon": (-122.65, -121.65)}
vmin, vmax = 50_000, 500_000
cmap = plt.get_cmap("YlOrRd")

def draw_panel(ax, res, extent, label):
    keep = panels[res][panels[res]["count"] >= 3]
    drawn = 0
    for cell, row in keep.iterrows():
        clat, clon = h3.cell_to_latlng(cell)
        if not (extent["lat"][0] <= clat <= extent["lat"][1]
                and extent["lon"][0] <= clon <= extent["lon"][1]):
            continue
        shade = (row["mean"] - vmin) / (vmax - vmin)
        ax.add_patch(MplPolygon([(lng, la) for la, lng in h3.cell_to_boundary(cell)],
                                closed=True, facecolor=cmap(np.clip(shade, 0, 1)),
                                edgecolor="none"))
        drawn += 1
    ax.set_xlim(*extent["lon"]); ax.set_ylim(*extent["lat"])
    ax.set_aspect(1 / math.cos(math.radians(np.mean(extent["lat"]))))
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"res {res} — {drawn:,} cells   ({label})", fontsize=11)

fig, axes = plt.subplots(2, 3, figsize=(15, 11))
for ax, res in zip(axes[0], (4, 5, 6)):
    draw_panel(ax, res, CA, "whole state")
for ax, res in zip(axes[1], (7, 8, 9)):
    draw_panel(ax, res, BAY, "Bay Area zoom")

fig.suptitle("California house values — the same 20,640 rows on six different grids\n"
             "top: the whole state   ·   bottom: zoomed to the Bay Area",
             y=1.00, fontsize=13)
plt.tight_layout(); plt.show()

Read the two rows together and the trade-off is plain.

**Across the state (top),** resolution 4 is a smooth blur — real, but it tells you nothing about
any particular place. By resolution 6 the Bay Area, Los Angeles and the Central Valley have
separated out. Note the cell *count* barely moves from res 7 to res 9 (1,925 → 1,998) even though
the cells shrink 50-fold: past a point, extra resolution stops buying coverage and only splits
the same points into more, emptier cells.

**Zoomed in (bottom),** resolution 7 is coherent, resolution 8 starts to speckle, and by
resolution 9 the map is mostly holes — each surviving cell is one small cluster of block groups,
and the neighbourhood signal has broken up.

This is the same story the hands-on lab tells with a model: error bottoms out at resolution 7 and
climbs again after. Here you can simply *see* it.

---
## Part 5 · So which resolution do I pick?

### The rule of thumb

> **Pick the finest resolution whose neighbourhood still holds enough members for a stable
> statistic — and watch COVERAGE, not cell size.**

Cell size is a property of the grid. Coverage is a property of *your data on that grid*, which
is what actually determines whether the feature works. Two datasets over the same city can want
different resolutions purely because one is denser.

In the hands-on lab this played out exactly: model error bottomed out at **resolution 7**, where
93% of rows still had at least one neighbour. At resolution 9 coverage collapsed to 53% and the
feature stopped working — while the *cells* were still perfectly reasonable objects.

In [ ]:
# ==========================================================================
# STEP 13 · The diagnostic to run on your own data, at every candidate res.
# --------------------------------------------------------------------------
# Coverage = the share of rows that have at least one OTHER row in their
# grid_disk(cell, 1) neighbourhood. No model, no target -- this is purely a
# property of your data's density, so you can run it before modelling anything.
#
# Read the output as: the finest resolution still above ~90% is your candidate.
#
# NOTE these numbers run slightly higher than the hands-on lab's sweep (93.8%
# vs 93.0% at res 7). Not a discrepancy: the lab builds neighbourhoods from
# TRAINING rows only, so it has a quarter fewer points to find. Here there is
# no model and no split, so every row counts. Use this version for scouting a
# dataset; expect real coverage to drop once you hold data out.
# ==========================================================================
scout = []
for res in range(4, 11):
    cl = np.array([h3.latlng_to_cell(a, o, res) for a, o in zip(lat_a, lon_a)])
    counts = pd.Series(cl).value_counts().to_dict()
    nbrs = np.array([sum(counts.get(d, 0) for d in h3.grid_disk(c, 1)) - 1 for c in cl])
    scout.append((res, float(np.median(nbrs)), 100 * float((nbrs > 0).mean())))

# the candidate is the FINEST resolution still clearing the bar, not every one
above = [s[0] for s in scout if s[2] >= 90.0]
pick = max(above) if above else None

print(f"{'res':>4}{'edge':>10}{'median nbrs':>14}{'coverage':>11}")
for res, med, cov in scout:
    flag = "   <-- finest above 90% : start here" if res == pick else ""
    print(f"{res:>4}{human_edge(h3.average_hexagon_edge_length(res, unit='km')):>10}"
          f"{med:>14.0f}{cov:>10.1f}%{flag}")

### Cheat sheet

| If you're working on... | Start at | Because |
|---|---|---|
| Country or state patterns | **4–5** | 1,770 / 253 km² — regions, not places |
| City-level demand, delivery zones | **6–7** | 36 / 5 km² — a town, a suburb |
| Neighbourhood analytics, pricing | **8–9** | 0.74 / 0.11 km² — blocks. **Start here if unsure** |
| Street-level, kerbside, buildings | **10–11** | 15,000 / 2,150 m² |
| Individual assets, sensors | **12+** | Sub-metre; usually more precision than your GPS has |

### Three things people get wrong

1. **Chasing precision.** A finer grid is not a better grid. Past the point where cells stop
   holding enough data, every extra resolution makes the analysis *worse* while looking more
   impressive. The lab's inverted U is the proof.
2. **Assuming cells are equal area.** They are not — the next cell measures it. "Average area"
   in Part 1 is genuinely an average.
3. **Forgetting the twelve pentagons.** Resolution 0 has 110 hexagons and 12 pentagons, and every
   finer resolution inherits them. They sit in the ocean by design, so most projects never meet
   one — but if you work on maritime or global data, test for them rather than being surprised.

In [ ]:
# ==========================================================================
# STEP 14 · Two honesty checks on claims people repeat about H3.
# --------------------------------------------------------------------------
# (a) "Hexagons are equal area." Measure the real spread at one resolution
#     across the latitudes our data actually spans.
# (b) "There are 12 pentagons." Count them, and confirm they persist at
#     finer resolutions rather than being a res-0 curiosity.
# ==========================================================================
print("(a) how equal is 'equal area'? -- res 7 cells across California")
areas = [h3.cell_area(h3.latlng_to_cell(la, -120.0, 7), unit="km^2")
         for la in (32.5, 37.0, 42.0)]
print(f"    areas: {[round(a, 4) for a in areas]} km2")
print(f"    spread across the state: {max(areas)/min(areas):.3f}x")
print("    -> real, but small. Don't build an argument for hexagons on it.")
print()
print("(b) pentagons")
res0 = h3.get_res0_cells()
print(f"    resolution 0: {len(res0)} cells, of which "
      f"{sum(1 for c in res0 if h3.is_pentagon(c))} are pentagons")
pent = next(c for c in res0 if h3.is_pentagon(c))
print(f"    a pentagon at res 0 : {pent} -> centre "
      f"{tuple(round(x, 3) for x in h3.cell_to_latlng(pent))}")
print(f"    its children at res 1: {len(h3.cell_to_children(pent, 1))} "
      f"(a hexagon has 7) -- pentagons have 6, and they persist all the way down")

---
## Where to go next

- **The hands-on lab** — `H3_Hands_On_Lab.ipynb`, where these resolutions get put to work and
  the choice is settled with a model rather than a rule of thumb.
- **`h3geo.org/docs`** — the official documentation, including the comparisons with S2, Geohash
  and QuadKey.
- **Run Part 4's diagnostic on your own data.** It takes one column of latitudes, one of
  longitudes, and about ten seconds — and it answers the resolution question better than any
  table can.